# Fazendo as features

## Importacao de dados

In [4]:
import sys
from pathlib import Path
import pandas as pd
from src.features_amr import AmblerClassifier, anotar_resistoma, build_features
import matplotlib.pyplot as plt
import argos
from argos import ArgosDataset

In [5]:
def encontrar_raiz_repo(marcadores=(".git", "src")):
    """Sobe o diretorio ate achar `.git` (repo real) ou `src/` (funciona tambem
    em sandbox local sem git inicializado) -- mesmo padrao usado nos notebooks
    da disciplina/repo original, generalizado pra cobrir os 2 casos."""
    caminho = Path.cwd()
    for pasta in [caminho, *caminho.parents]:
        if any((pasta / m).exists() for m in marcadores):
            return pasta
    raise FileNotFoundError(f"Nao achei nenhum de {marcadores} subindo a partir de {caminho}")


RAIZ = encontrar_raiz_repo()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

In [6]:
print(RAIZ)

/LACTAS-HELISSON-01/rodrigo.lusa/ml_isolados_brasileiros/notebooks


In [27]:
metadata = pd.read_csv("../data/processed/metadata_filtrada.csv")
ids_finais = metadata["sample"].tolist()
ids_finais

['GCA_000216055.2',
 'GCA_000316425.1',
 'GCA_000223095.2',
 'GCA_036761135.1',
 'GCA_003670255.1',
 'GCA_006334885.1',
 'GCA_006334895.1',
 'GCA_024700995.1',
 'GCA_024701035.1',
 'GCF_000363915.2',
 'GCA_000348045.1',
 'GCA_000348065.1',
 'GCA_000327825.2',
 'GCA_000328125.2',
 'GCA_000327845.2',
 'GCA_000327865.2',
 'GCA_021538905.1',
 'GCA_000355215.1',
 'GCA_000355195.1',
 'GCA_000355235.1',
 'GCA_000355255.1',
 'GCA_000512165.1',
 'GCA_041901425.1',
 'GCA_023093655.1',
 'GCA_026223635.2',
 'GCA_000386205.2',
 'GCA_000386225.2',
 'GCA_000386745.2',
 'GCA_000386765.2',
 'GCA_003859745.1',
 'GCA_018224045.1',
 'GCA_000306435.2',
 'GCA_000346675.1',
 'GCA_000306135.2',
 'GCA_000350585.1',
 'GCA_000369605.1',
 'GCA_000346655.1',
 'GCA_011463375.1',
 'GCA_030668605.1',
 'GCA_031192385.1',
 'GCA_028067445.1',
 'GCA_020622285.1',
 'GCA_025823245.1',
 'GCA_002102185.1',
 'GCF_000647855.2',
 'GCF_001921685.1',
 'GCF_001921695.1',
 'GCA_002260005.1',
 'GCA_002260015.1',
 'GCA_002260065.1',


In [28]:
dataset = ArgosDataset.from_parquet("../data/argos_project/parquet_data", samples=ids_finais)
dataset.resistome.head()

,ORF_ID,Cut_Off,Pass_Bitscore,Best_Hit_Bitscore,Best_Identities,drug_class,mechanism,amr_gene_family,Predicted_Protein,CARD_Protein_Sequence,...,gene,aro_accession,model_type,gene_clean,seq_name,start,end,strand,length_nt,contig_type
0,ABSC01000258.1_1 # 45 # 395 # 1 # ID=83_1;part...,Strict,100,115.931,42.98,{phosphonic acid antibiotic},{antibiotic inactivation},{fosfomycin thiol transferase},MLNTLFDAKEIYDSQQKKFSLYPEKFFLVKDLWIAVMQNSSNKLPK...,MKGISHITFIVRDLNRMAALLCEGLGAREVYDSSNQNFSLSREKFF...,...,FosI,ARO:3007370,protein homolog model,FosI,ABSC01000258.1,45,395,1,351,chromosome
1,ABSC01000180.1_7 # 6357 # 7019 # -1 # ID=161_7...,Strict,50,220.705,57.14,{glycopeptide antibiotic},{antibiotic target alteration},"{glycopeptide resistance gene cluster, vanY}",MKRSYKTVAVILLIVLLASIGLFFRKTPQKIQVCGEERDSWNLLLV...,MEKSNYHSNVNHHKRHMKQSGEKRAFLWAFIISFTVCTLFLGWRLV...,...,vanY gene in vanB cluster,ARO:3002956,protein homolog model,vanY_in_vanB_cl,ABSC01000180.1,6357,7019,-1,663,chromosome
2,ABSC01000175.1_1 # 81 # 1520 # -1 # ID=166_1;p...,Strict,900,966.066,99.79,{aminoglycoside antibiotic},{antibiotic inactivation},{aminoglycoside bifunctional resistance protein},MNIVENEICIRTLIDDDFPLMLKWLTDERVLEFYDGRDKKYTLESL...,MNIVENEICIRTLIDDDFPLMLKWLTDERVLEFYGGRDKKYTLESL...,...,AAC(6')-Ie-APH(2'')-Ia bifunctional protein,ARO:3002597,protein homolog model,AAC6_Ie_APH2_Ia,ABSC01000175.1,81,1520,-1,1440,chromosome
3,ABSC01000090.1_4 # 3146 # 4876 # 1 # ID=251_4;...,Strict,960,989.949,82.81,"{macrolide antibiotic, fluoroquinolone antibio...",{antibiotic efflux},{ATP-binding cassette (ABC) antibiotic efflux ...,MKLMWHYTMRYKKFLFFNFICVFGFILIELGLPTILARMIDVGIRN...,MKLMWRYTMRYKKLLFADFICVFGFILIELGLPTILARMIDKGIIP...,...,efrA,ARO:3003948,protein homolog model,efrA,ABSC01000090.1,3146,4876,1,1731,chromosome
4,ABSC01000050.1_4 # 2667 # 3404 # 1 # ID=291_4;...,Strict,400,489.574,97.96,"{lincosamide antibiotic, streptogramin B antib...",{antibiotic target alteration},{Erm 23S ribosomal RNA methyltransferase},MNKNIKYSQNFLTSEKVLNQIIKQLNLKETDTVYEIGTGKGHLTTK...,MNKNIKYSQNFLTSEKVLNQIIKQLNLKETDTVYEIGTGKGHLTTK...,...,ErmB,ARO:3000375,protein homolog model,ErmB,ABSC01000050.1,2667,3404,1,738,plasmid


## Ajustando dados

In [40]:
RECORTE = metadata["WHO_Priority"] == "Critical"
metadata_recorte = metadata[RECORTE].set_index('sample')

metadata_recorte.head()

,Species,Source,Date,Location,BioSample,Estado,Região,source_type,WHO_Priority
sample,,,,,,,,,
GCA_000316425.1,Escherichia coli,NaN,1990.0,Brazil,SAMN01041333,NaN,NaN,Unknown,Critical
GCA_000355215.1,Escherichia coli,stool,1998.0,Envira,SAMN00829025,AM,Norte,Gastrointestinal,Critical
GCA_000355195.1,Escherichia coli,stool,1998.0,Envira,SAMN00829717,AM,Norte,Gastrointestinal,Critical
GCA_000355235.1,Escherichia coli,stool,1998.0,Brazil,SAMN00829297,NaN,NaN,Gastrointestinal,Critical
GCA_000355255.1,Escherichia coli,stool,1998.0,Brazil,SAMN00829299,NaN,NaN,Gastrointestinal,Critical


In [43]:
ids_recorte = metadata_recorte.index
print(len(ids_recorte))
print(ids_recorte)

2230
Index(['GCA_000316425.1', 'GCA_000355215.1', 'GCA_000355195.1',
       'GCA_000355235.1', 'GCA_000355255.1', 'GCA_000512165.1',
       'GCA_041901425.1', 'GCA_023093655.1', 'GCA_026223635.2',
       'GCA_018224045.1',
       ...
       'GCA_016074895.1', 'GCA_015681415.1', 'GCA_048966275.1',
       'GCA_048966225.1', 'GCA_054551795.1', 'GCA_054551815.1',
       'GCA_054551855.1', 'GCA_054551875.1', 'GCA_054551965.1',
       'GCA_054612235.1'],
      dtype='object', name='sample', length=2230)


In [44]:
resistome_recorte = dataset.resistome[dataset.resistome["sample_id"].isin(ids_recorte)].reset_index(drop=True)

print(f"{len(metadata_recorte)} genomas no recorte, {len(resistome_recorte)} hits de AMR")

2230 genomas no recorte, 105873 hits de AMR


## Fazendo classificacao de beta-lactamase

In [45]:
classifier = AmblerClassifier("../data/raw/refgenes.tsv")

In [46]:
resistome_anotado = anotar_resistoma(resistome_recorte, classifier)

resistome_anotado.head()

,ORF_ID,Cut_Off,Pass_Bitscore,Best_Hit_Bitscore,Best_Identities,drug_class,mechanism,amr_gene_family,Predicted_Protein,CARD_Protein_Sequence,...,start,end,strand,length_nt,contig_type,ambler_class,tem_carbapenem,tem_cefalosporina,carbapenemase_confirmada,toca_beta_lactam
0,ADUN01000196.1_41 # 41179 # 42273 # 1 # ID=4_4...,Strict,250,266.544,39.50,{glycopeptide antibiotic},{antibiotic target alteration},"{glycopeptide resistance gene cluster, Van lig...",MEKLRVGIVFGGKSAEHEVSLQSAKNIVDAIDKSRFDVVLLGIDKQ...,MQNKKIAVIFGGNSTEYEVSLQSASAVFENINTNKFDIIPIGITRS...,...,41179,42273,1,1095,chromosome,None,False,False,False,False
1,ADUN01000193.1_83 # 80914 # 82557 # -1 # ID=7_...,Strict,1050,1103.200,99.45,{peptide antibiotic},{antibiotic efflux},{ATP-binding cassette (ABC) antibiotic efflux ...,MELLVLVWRQYRWPFISVMALSLASAALGIGLIAFINQRLIETADT...,MELLVLVWRQYRWPFISVMALSLASAALGIGLIAFINQRLIETADT...,...,80914,82557,-1,1644,chromosome,None,False,False,False,False
2,ADUN01000192.1_7 # 6574 # 7821 # 1 # ID=8_7;pa...,Strict,725,829.321,99.28,{aminocoumarin antibiotic},{antibiotic efflux},{resistance-nodulation-cell division (RND) ant...,MKGSYKSRWVIVIVVVIAAIAAFWFWQGRNDSRSAAPGATKQAQQS...,MKGSYKSRWVIVIVVVIAAIAAFWFWQGRNDSRSAAPGATKQAQQS...,...,6574,7821,1,1248,plasmid,None,False,False,False,False
3,ADUN01000192.1_8 # 7821 # 10943 # 1 # ID=8_8;p...,Strict,1800,2075.060,99.52,{aminocoumarin antibiotic},{antibiotic efflux},{resistance-nodulation-cell division (RND) ant...,MQVLPPSSTGGPSRLFIMRPVSTTLLMVAILLAGIIGYRALPVSAL...,MQVLPPSSTGGPSRLFIMRPVATTLLMVAILLAGIIGYRALPVSAL...,...,7821,10943,1,3123,plasmid,None,False,False,False,False
4,ADUN01000192.1_9 # 10944 # 14021 # 1 # ID=8_9;...,Strict,1800,2063.110,99.61,{aminocoumarin antibiotic},{antibiotic efflux},{resistance-nodulation-cell division (RND) ant...,MKFFALFIYRPVATILLSVAITLCGILGFRMLPVAPLPQVDFPVIM...,MKFFALFIYRPVATILLSVAITLCGILGFRMLPVAPLPQVDFPVII...,...,10944,14021,1,3078,plasmid,None,False,False,False,False


In [47]:
n_classificados = resistome_anotado["ambler_class"].notna().sum()
print(f"{n_classificados}/{len(resistome_anotado)} hits receberam classe de Ambler")

8593/105873 hits receberam classe de Ambler


## Fazendo as features (por genoma)

`build_features` faz tudo de uma vez -- os 5 grupos de beta-lactamase (como proporcao do total
beta-lactamico), mobilidade plasmidial e as 2 contagens brutas (aminoglicosideo/polimixina).\

In [48]:
raw_features = build_features(resistome_anotado, metadata_recorte.index)

print(raw_features.shape)
raw_features.describe()


(2230, 9)


,n_beta_lactam_total,pct_esbl,pct_A_serino_carbapenemase,pct_BD_carbapenemase,pct_C_ampc,pct_nao_hidrolise,pct_beta_lactam_plasmidial,n_aminoglicosideo,n_polimixina
count,2230.000000,2230.000000,2230.000000,2230.000000,2230.000000,2230.000000,2230.000000,2230.000000,2230.000000
mean,14.526906,0.115218,0.030265,0.059858,0.048992,0.724002,0.231096,3.441256,2.556951
std,4.637892,0.098194,0.041172,0.112422,0.059917,0.145555,0.179681,2.802598,1.391873
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,13.000000,0.000000,0.000000,0.000000,0.000000,0.600000,0.071429,2.000000,2.000000
50%,16.000000,0.125000,0.000000,0.000000,0.000000,0.750000,0.212406,3.000000,3.000000
75%,17.000000,0.176471,0.062500,0.058824,0.086051,0.833333,0.375000,5.000000,3.000000
max,42.000000,1.000000,0.333333,0.500000,0.333333,1.000000,0.687500,48.000000,6.000000


## GC%

In [53]:
gc = pd.read_csv("../data/raw/gc_stats.txt", sep=r"\s+", thousands=",")
gc.head()

,file,format,type,num_seqs,sum_len,min_len,avg_len,max_len,Q1,Q2,Q3,sum_gap,N50,N50_num,Q20(%),Q30(%),AvgQual,GC(%),sum_n
0,genomes_contig_type/GCA_000172875.1_ASM17287v1...,FASTA,DNA,293,2486720,216,8487.1,150593,996.0,2832.0,9453.0,0,22620,28,0,0,0,37.94,0.0
1,genomes_contig_type/GCA_000172875.1_ASM17287v1...,FASTA,DNA,34,364754,2531,10728.1,36405,4772.0,9614.5,12656.0,0,12656,9,0,0,0,36.24,0.0
2,genomes_contig_type/GCA_000188755.2_ASM18875v2...,FASTA,DNA,116,4411386,277,38029.2,222066,2212.5,19067.5,51921.0,0,91542,17,0,0,0,50.74,4.0
3,genomes_contig_type/GCA_000188755.2_ASM18875v2...,FASTA,DNA,36,744317,954,20675.5,90331,7710.0,13332.5,26628.5,0,29539,7,0,0,0,49.15,1.0
4,genomes_contig_type/GCA_000188815.2_ASM18881v2...,FASTA,DNA,46,4960945,234,107846.6,730778,683.0,41411.5,198155.0,0,249753,7,0,0,0,50.84,0.0


In [56]:
# Conteúdo GC em contigs cromossomais vs plasmidiais
gc["sample_id"] = gc["file"].str.extract(r"(GC[AF]_\d+\.\d+)")
gc["contig_type"] = gc["file"].str.extract(r"_(chromosome|plasmid)\.fna$")

gc_wide = gc.pivot(index="sample_id", columns="contig_type", values="GC(%)")
gc_wide.columns = ["gc_cromossomal", "gc_plasmidial"]
gc_wide["gc_diff_plasmid_cromossomo"] = gc_wide["gc_plasmidial"] - gc_wide["gc_cromossomal"]
gc_wide = gc_wide.reindex(metadata_recorte.index)

gc_wide.head()

,gc_cromossomal,gc_plasmidial,gc_diff_plasmid_cromossomo
sample,,,
GCA_000316425.1,50.57,47.65,-2.92
GCA_000355215.1,50.82,48.72,-2.10
GCA_000355195.1,50.81,49.31,-1.50
GCA_000355235.1,50.75,49.62,-1.13
GCA_000355255.1,50.66,49.49,-1.17


In [59]:
raw_features = raw_features.join(
    gc_wide["gc_diff_plasmid_cromossomo"]
)

raw_features.head()

,n_beta_lactam_total,pct_esbl,pct_A_serino_carbapenemase,pct_BD_carbapenemase,pct_C_ampc,pct_nao_hidrolise,pct_beta_lactam_plasmidial,n_aminoglicosideo,n_polimixina,gc_diff_plasmid_cromossomo
sample,,,,,,,,,,
GCA_000316425.1,15,0.000000,0.0,0.0,0.066667,0.933333,0.133333,0,4,-2.92
GCA_000355215.1,16,0.000000,0.0,0.0,0.062500,0.937500,0.125000,2,3,-2.10
GCA_000355195.1,16,0.000000,0.0,0.0,0.062500,0.937500,0.125000,2,3,-1.50
GCA_000355235.1,17,0.058824,0.0,0.0,0.058824,0.882353,0.058824,2,4,-1.13
GCA_000355255.1,17,0.058824,0.0,0.0,0.058824,0.882353,0.176471,0,4,-1.17


## Fazendo a tabela final

In [60]:
raw_features.to_csv("../data/processed/features.csv")
print(f"Salvo: {'../data/processed/features.csv'} -- {raw_features.shape}")
raw_features.head()


Salvo: ../data/processed/features.csv -- (2230, 10)


,n_beta_lactam_total,pct_esbl,pct_A_serino_carbapenemase,pct_BD_carbapenemase,pct_C_ampc,pct_nao_hidrolise,pct_beta_lactam_plasmidial,n_aminoglicosideo,n_polimixina,gc_diff_plasmid_cromossomo
sample,,,,,,,,,,
GCA_000316425.1,15,0.000000,0.0,0.0,0.066667,0.933333,0.133333,0,4,-2.92
GCA_000355215.1,16,0.000000,0.0,0.0,0.062500,0.937500,0.125000,2,3,-2.10
GCA_000355195.1,16,0.000000,0.0,0.0,0.062500,0.937500,0.125000,2,3,-1.50
GCA_000355235.1,17,0.058824,0.0,0.0,0.058824,0.882353,0.058824,2,4,-1.13
GCA_000355255.1,17,0.058824,0.0,0.0,0.058824,0.882353,0.176471,0,4,-1.17
